Imlo coursework

In [13]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

In [14]:
# code to use my gpu
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(device)

cuda


In [15]:
transform = transforms.Compose([
    # adding random augmentations
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),


    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

In [16]:
#defining the train and val
train_data = torchvision.datasets.OxfordIIITPet(
    root = './data',
    split = 'trainval',
    transform = transform,
    download = True
)

#splitting trainval
train_size = int(0.8 * len(train_data))
val_size = len(train_data) - train_size
train_data, val_data = torch.utils.data.random_split(
    train_data,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42) #makes sure that the images used in train and val stick tg
)

#dataloaders for train and val
train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
val_loader = torch.utils.data.DataLoader(val_data, batch_size=32, shuffle=True, num_workers=2)

In [17]:
image, label = train_data[0]

In [18]:
image.size()

torch.Size([3, 128, 128])

In [19]:
# Adding names for the catergories
class_names = train_data.dataset.classes

In [20]:
# Defining the layers
class NeuralNet(nn.Module):
    def __init__(self):
    #calls constructor from nn.Module
        super().__init__()

        self.conv1 = nn.Conv2d(3, 12, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(12, 24, 5)

        self.fc1 = nn.Linear(24 * 29 * 29, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 37)

    def forward(self, input):
        input = self.pool(F.relu(self.conv1(input)))  #applying conv1, then RELU, then pooling layer
        input = self.pool(F.relu(self.conv2(input)))  #applying conv2, then RELU then pooling layer
        input = torch.flatten(input, 1)  #flattening
        input = F.relu(self.fc1(input))  #applying fc1, then RELU
        input = F.relu(self.fc2(input))  #applying fc2, then RELU
        input = self.fc3(input)  #applying fc3
        return input

In [21]:
# defining the NN itself
network = NeuralNet().to(device)
loss_func = nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(network.parameters(), lr=0.001)

In [22]:
# training the model
for epoch in range(30):
    print("Training epoch:", epoch)
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimiser.zero_grad()

        outputs = network(inputs)
        loss = loss_func(outputs, labels)
        loss.backward()
        optimiser.step()

        running_loss += loss.item()

    running_loss_calc = running_loss / len(train_loader)
    print("Loss:", running_loss_calc)

Training epoch: 0
Loss: 3.6147205155828726
Training epoch: 1
Loss: 3.513811787833338
Training epoch: 2
Loss: 3.365290665108225
Training epoch: 3
Loss: 3.2260388213655222
Training epoch: 4
Loss: 3.130153855551844
Training epoch: 5
Loss: 3.0123753444008203
Training epoch: 6
Loss: 2.9044255406960198
Training epoch: 7
Loss: 2.79767403136129
Training epoch: 8
Loss: 2.677319218283114
Training epoch: 9
Loss: 2.5612685848837313
Training epoch: 10
Loss: 2.436980883712354
Training epoch: 11
Loss: 2.308777982773988
Training epoch: 12
Loss: 2.2000251088453378
Training epoch: 13
Loss: 2.063593583262485
Training epoch: 14
Loss: 1.9177238228528395
Training epoch: 15
Loss: 1.7769806696021038
Training epoch: 16
Loss: 1.7261328632416932
Training epoch: 17
Loss: 1.6020064587178438
Training epoch: 18
Loss: 1.4194801164710003
Training epoch: 19
Loss: 1.340364212575166
Training epoch: 20
Loss: 1.2380855452755224
Training epoch: 21
Loss: 1.2232911981966184
Training epoch: 22
Loss: 1.0840064854077671
Training

In [23]:
# testing the model on val data
correct = 0
total = 0

network.eval()

with torch.no_grad():
  for images, labels in val_loader:

    images = images.to(device)
    labels = labels.to(device)


    outputs = network(images)
    predicted = outputs.argmax(1)

    total += len(labels)
    correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print("Accuracy:", accuracy)

Accuracy: 15.217391304347826
